# Sixth LAB

### Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.

### Project - Hotel AI Assistant (SLM version)

In [1]:
# imports
import os
import json
import sqlite3
import gc
import re
import inspect
from threading import Thread
import torch
import gradio as gr
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

In [2]:
# Check for GPU / CPU
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA: 12.1
Using device: cuda


In [3]:
# Log in to HuggingFace (Llama is a gated model)
load_dotenv(override=True)

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# Define model ID
model_id = "meta-llama/Llama-3.2-3B-Instruct"

DECODER = json.JSONDecoder()
MAX_TOOL_ROUNDS = 5
CUMULATIVE = False
PRICE_WORDS = ("price", "cost", "how much", "tariff", "room rate", "stay price", "accommodation rate")

### Load Model

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map=device,
    low_cpu_mem_usage=True,
    dtype=torch.float16,
)

print("Model loaded successfully!")
print("Memory footprint:", round(model.get_memory_footprint() / 1024**3, 2), "GB")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!
Memory footprint: 5.98 GB


In [6]:
# Define system message
system_message = """
You are HotelAI, a helpful hotel information assistant.

Your responsibilities:

1. Answer hotel-related questions clearly.
2. Keep answers concise and conversational.
3. When the user explicitly asks for hotel prices, use the get_hotel_price tool.
4. Never invent hotel prices.
5. If the destination is missing, ask the user which city they mean.
6. If the requested city is not supported by the database, explain that price information is unavailable.
7. Do not call tools for greetings, general questions, recommendations or hotel availability questions.
8. Never invent information returned by a tool.
9. After receiving a tool result, explain it naturally.

Keep responses to 1-3 sentences.
"""

## Tools

We give the model a function description. The model can't run code itself - it replies with a JSON "tool call" asking *us* to run the function, then we feed the result back.

In [7]:
# Database of hotel prices
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (hotel TEXT PRIMARY KEY, price REAL)')
    conn.commit()


def get_hotel_price(hotel):
    print(f"DATABASE TOOL CALLED: Getting price for {hotel}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE hotel = ?', (hotel.lower(),))
        result = cursor.fetchone()
        return f"Stay price in {hotel} is Rs {result[0]}" if result else f"No price data available for hotel in {hotel}"


def set_hotel_price(hotel, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (hotel, price) VALUES (?, ?) ON CONFLICT(hotel) DO UPDATE SET price = ?', (hotel.lower(), price, price))
        conn.commit()


hotel_prices = {"New Delhi": 5000, "Bangalore": 8000, "Kochi": 6500, "Mumbai": 10000, "Shimla": 5500}
for hotel, price in hotel_prices.items():
    set_hotel_price(hotel, price)

TOOL_REGISTRY = {"get_hotel_price": get_hotel_price}

get_hotel_price("Bangalore")

DATABASE TOOL CALLED: Getting price for Bangalore


'Stay price in Bangalore is Rs 8000.0'

In [8]:
# Function description, in the structure the chat template expects
price_function = {
    "name": "get_hotel_price",
    "description": "Get the price of a stay in your destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "hotel": {
                "type": "string",
                "description": "The city that the customer wants to stay in",
            },
        },
        "required": ["hotel"],
    }
}

tools = [{"type": "function", "function": price_function}]

### Helpers

Llama 3.2 emits a tool call as plain text, e.g. `{"name": "get_ticket_price", "parameters": {"destination_city": "London"}}`, so we parse it ourselves. Its chat template only allows **one** tool call per assistant message, so we handle one call per round and loop.

In [9]:
def wants_tools(prompt):
    p = prompt.lower()
    return any(w in p for w in PRICE_WORDS)

def valid_call(call, context):
    """Reject calls whose argument wasn't actually mentioned by the user."""
    if call["name"] not in TOOL_REGISTRY:
        return False
    vals = [str(v).strip().lower() for v in call["arguments"].values()]
    return bool(vals) and all(v and v in context for v in vals)

def to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", "")
    if isinstance(content, list):
        return "".join(to_text(p) for p in content)
    return str(content)

TOOL_CALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)

In [10]:
def parse_tool_calls(text):
    """Handle Llama-style JSON and Qwen/Hermes <tool_call> blocks."""
    text = text.replace("<|python_tag|>", "").replace("<tool_call>", "").replace("</tool_call>", "")
    calls, i = [], 0
    while True:
        i = text.find("{", i)
        if i == -1:
            break
        try:
            obj, end = DECODER.raw_decode(text, i)
        except json.JSONDecodeError:
            i += 1
            continue
        if isinstance(obj, dict) and "name" in obj and isinstance(obj["name"], str):
            args = obj.get("parameters", obj.get("arguments", {}))
            if isinstance(args, str):
                try:
                    args = json.loads(args)
                except json.JSONDecodeError:
                    args = {}
            calls.append({"name": obj["name"], "arguments": args})
        i = end
    return calls

def looks_like_tool_call(raw):
    s = raw.lstrip()
    return s.startswith(("{", "<tool_call>", "<|python_tag|>"))

def run_tool(name, arguments):
    fn = TOOL_REGISTRY.get(name)
    if fn is None:
        return f"Unknown tool {name}"
    params = list(inspect.signature(fn).parameters)
    # keep only args the function accepts
    kwargs = {k: v for k, v in arguments.items() if k in params}
    
    # if the model used a different key name and there's exactly one param, map it
    if not kwargs and len(params) == 1 and len(arguments) == 1:
        kwargs = {params[0]: next(iter(arguments.values()))}
    try:
        return fn(**kwargs)
    except Exception as e:
        return f"Tool error: {e}"
    
def generate_stream(messages, tools=None, max_new_tokens=250):

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    inputs = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    generation_kwargs = {
        **inputs,
        "max_new_tokens": max_new_tokens,
        "streamer": streamer,
        "do_sample": False,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id
    }

    errors = []

    def generate():
        try:
            with torch.inference_mode():
                model.generate(**generation_kwargs)
        except Exception as e:
            errors.append(e)
            streamer.end()

    thread = Thread(target=generate, daemon=True)
    thread.start()

    try:
        for chunk in streamer:
            # Yield ONLY the newly generated text
            yield chunk

    finally:
        thread.join()

    if errors:
        raise RuntimeError(f"Generation failed: {errors[0]}")

In [11]:
# Handle tool calls
def model_call(prompt, history):

    # Defining history & Prompt
    history = [{"role": h["role"], "content": to_text(h["content"])} for h in history]

    active = tools if wants_tools(prompt) else None

    sys_msg = system_message
    if active:
        sys_msg += """
            You have access to the get_hotel_price database tool.

            When the user asks for a stay price and provides a city,
            call the tool.

            After receiving database results:
            - Use the exact price returned by the database.
            - Never say that price information is unavailable
            if the database returned a price.
            - Never invent or modify prices.
            - Answer naturally and concisely.
        """

    messages = [
        {"role": "system", "content": sys_msg}] + history + [{"role": "user", "content": prompt}]

    context = " ".join([h["content"] for h in history if h["role"] == "user"] + [prompt]).lower()

    print(context)

    tool_results = []
    tools_done = False

    for round_idx in range(MAX_TOOL_ROUNDS + 1):
        is_last = round_idx == MAX_TOOL_ROUNDS

        # After a tool has run, force a plain-text answer
        round_tools = None if (tools_done or is_last or not active) else active

        raw = ""
        for chunk in generate_stream(messages, round_tools):
            raw += chunk
            if looks_like_tool_call(raw):
                continue
            if raw.strip():
                yield raw

        # print(f"[round {round_idx}] tools={'on' if round_tools else 'off'} raw={raw!r}", flush=True)

        calls = parse_tool_calls(raw) if round_tools else []

        if not calls:
            if not raw.strip():
                if tool_results:
                    yield "\n".join(tool_results)      # fall back to the raw tool result
            return

        good = [c for c in calls if valid_call(c, context)]
        if not good:
            active = None

            messages.append({
                "role": "assistant",
                "content": raw
            })

            messages.append({
                "role": "user",
                "content": (
                    "Please respond normally in plain text. "
                    "Do not generate a tool call."
                )
            })

            continue

        yield "🔧 Looking up price...\n\n"

        # good = good[:1]

        # for c in good:
        #     result = run_tool(c["name"], c["arguments"])
        #     result = str(result)
        #     tool_results.append(result)

        #     print(f"DATABASE RESULT: {result}", flush=True)

        #     # Handle city not found directly
        #     if "No price data available" in result:

        #         yield (f"Sorry, I don't have price information "
        #             f"for {c['arguments'].get('hotel', 'that city')}. "
        #             "Please try another destination."
        #         )

        #         return

        #     # Handle database errors
        #     if result.startswith("Tool error:"):
        #         yield "Sorry, I encountered an error while checking the database."
        #         return
    
        messages.append({
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {"type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}}
                for c in good
            ],
        })
        
        for c in good:
            result = run_tool(c["name"], c["arguments"])
            tool_results.append(str(result))
            messages.append({"role": "tool", "name": c["name"], "content": str(result)})

        tools_done = True

### FlightAI

In [ ]:
gr.ChatInterface(fn=model_call, 
                 title="HotelAI Assistant" 
                 ).\
                launch(theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="purple"))

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


hello
hello i want to know stay price in shimla
DATABASE TOOL CALLED: Getting price for Shimla
hello i want to know stay price in shimla thanks


In [ ]:
# Commands to clear GPU
del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()